# Phase 4: Data Cleaning and Validation

## Objective
This purpose of this phase is to prepare the Walmart sales dataset for exploratory data analysis and machine learning.

This phase will:

- Load a fresh copy of the raw dataset
- Preserving the original raw file
- Standardising column names
- Converting the date cloumn to datetime
- Validating required columns
- Checking missing values
- Checking duplicate records
- Validating categorical values
- Checking impossible numerical values
- Sorting the data chronologically
- Saving the cleaned dataset in `data/processed/` 

## Data Cleaning Approach

The original dataset will not be modified directly.

A fresh copy will be loaded from the raw-data directory, cleaned in memory, validated, and saved as a separated processed file.

This approach preserves the original data and makes the project reproducible.

In [31]:
# Importing the libraries

from pathlib import Path
import pandas as pd

In [32]:
# Defining the file paths

current_directory = Path.cwd()

project_root = (
    current_directory.parent
    if current_directory.name.lower() == "notebooks"
    else current_directory
)

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "Walmart.csv"
)

processed_data_path = (
    project_root
    / "data"
    / "procesed"
    / "walmart_cleaned.csv"

)

print("Raw dataset:", raw_data_path)
print("Processed dataset:", processed_data_path)

Raw dataset: c:\Users\Udoch\walmart-sales-regression\data\raw\Walmart.csv
Processed dataset: c:\Users\Udoch\walmart-sales-regression\data\procesed\walmart_cleaned.csv


In [33]:
# Confirming the raw file exists

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"The raw dataset was not found at: {raw_data_path}"
    )

print("Raw dataset found successfully.")

Raw dataset found successfully.


In [34]:
# Loading a fresh copy of the raw dataset

df_raw = pd.read_csv(raw_data_path)

df = df_raw.copy()

print("Raw dataset loaded successfully.")
print("Working copy created successfully.")

Raw dataset loaded successfully.
Working copy created successfully.


### Why a Working Copy Is Used

`df_raw` stores the original dataset loaded from the raw CSV file.

`df` is a separate working copy used for cleaning and transformation.

This prevents accidental modification of the original raw DataFrame during the cleaning process.

In [35]:
# Comparing the initial shape

initial_rows = df.shape[0]
initial_columns = df.shape[1]

print("Initial rows:", initial_rows)
print("Initial columns:", initial_columns)

Initial rows: 6435
Initial columns: 8


## Inspect Original Column Names

Column names should be consistent, readable, and suitable for Python code.

The original column names will be reviewed before standardisation.

In [36]:
# Inspecting the original column names

df.columns.tolist()

['Store',
 'Date',
 'Weekly_Sales',
 'Holiday_Flag',
 'Temperature',
 'Fuel_Price',
 'CPI',
 'Unemployment']

In [37]:
# Standardising the column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df.columns.tolist()

['store',
 'date',
 'weekly_sales',
 'holiday_flag',
 'temperature',
 'fuel_price',
 'cpi',
 'unemployment']

### Column-Name Interpretation

The column names were converted to lowercase snake case.

This improves readability and creates a consistent naming convention for later Python code, machine learning pipelines, and deployment files.

## Required Column Validation

The dataset must contain all expected columns before cleaning and modelling can continue.

In [38]:
# Validating the required columns

required_columns = [
    "store",
    "date",
    "weekly_sales",
    "holiday_flag",
    "temperature",
    "fuel_price",
    "cpi",
    "unemployment",
]

missing_required_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

unexpected_columns = [
    column
    for column in df.columns
    if column not in required_columns
]

print("Missing required columns:", missing_required_columns)
print("Unexpected columns:", unexpected_columns)

Missing required columns: []
Unexpected columns: []


### Required-Column Interpretation

No required columns are missing, and no unexpected columns are present.

The dataset structure matches the expected project schema.

## Converting the Date Column

The `date` column is currently stored as text.

It must be converted to a datetime data type so that the records can be sorted chronologically and used for time-based feature engineering.

In [39]:
# Converting the date column

df["date"] = pd.to_datetime(
    df["date"],
    format="%d-%m-%Y",
    errors="coerce",
)

print("Date conversion completed.")

Date conversion completed.


In [40]:
# Checking for invalid dates after conversion

invalid_date_count = df["date"].isna().sum()

print("Invalid dates:", invalid_date_count)

Invalid dates: 0


### Date-Conversion Interpretation

The `date` column was successfully converted from text to datetime.

No invalid dates were detected, which confirms that the original date values followed the expected day-month-year format.

In [41]:
# Confirming the data types

df.dtypes

store                    int64
date            datetime64[us]
weekly_sales           float64
holiday_flag             int64
temperature            float64
fuel_price             float64
cpi                    float64
unemployment           float64
dtype: object

### Data-Type Validation

The `date` column is now stored as a datetime variable.

The sales, temperature, fuel price, CPI, and unemployment columns remain numerical.

`store` and `holiday_flag` remain stored as integers, but they will later be treated as categorical features during preprocessing.

## Missing-Value Validation

Missing values are checked again after date conversion because invalid date values would have been converted to missing values.

In [42]:
# Checking the missing values again

missing_summary = pd.DataFrame(
    {
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        ).round(2),
    }
)

missing_summary

,missing_count,missing_percentage
store,0,0.0
date,0,0.0
weekly_sales,0,0.0
holiday_flag,0,0.0
temperature,0,0.0
fuel_price,0,0.0
cpi,0,0.0
unemployment,0,0.0


### Missing-Value Interpretation

No missing values were detected after the cleaning operations.

Therefore, no rows need to be removed and no missing-value imputation is required at this stage.

## Exact Duplicate Validation

Exact duplicate rows may cause repeated observations to influence model training unfairly.

In [43]:
# Checking exact duplicate rows

exact_duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicate_count)

Exact duplicate rows: 0


In [44]:
# Checking the duplicate store-date records

store_date_duplicate_count = (
    df
    .duplicated(
        subset=["store", "date"]
    )
    .sum()
)

print(
    "Duplicate store-date records:",
    store_date_duplicate_count,
)

Duplicate store-date records: 0


### Duplicate Validation Interpretation

No exact duplicate rows were found.

No duplicate combinations of `store` and `date` were found, which confirms that each store has only one record for each weekly date.

In [45]:
# Removing duplicates safely if any exist

df = (
    df
    .drop_duplicates()
    .drop_duplicates(
        subset=["store", "date"],
        keep="first",
    )
    .reset_index(drop=True)
)

print("Rows after duplicate handling:", df.shape[0])

Rows after duplicate handling: 6435


## Store Identifier Validation

Store identifiers should be positive whole numbers representing valid Walmart stores.

In [46]:
# Validating store identifiers

invalid_store_count = (
    (df["store"] <= 0)
    | (df["store"].isna())
).sum()

print("Invalid store identifiers:", invalid_store_count)
print("Minimum store ID:", df["store"].min())
print("Maximum store ID:", df["store"].max())
print("Number of stores:", df["store"].nunique())

Invalid store identifiers: 0
Minimum store ID: 1
Maximum store ID: 45
Number of stores: 45


## Holiday Flag Validation

The `holiday_flag` column should contain only:

- `0` for a non-holiday week
- `1` for a holiday week

In [47]:
holiday_values = sorted(
    df["holiday_flag"]
    .dropna()
    .unique()
    .tolist()
)

valid_holiday_values = {0, 1}

invalid_holiday_values = (
    set(holiday_values)
    - valid_holiday_values
)

print("Holiday values:", holiday_values)
print(
    "Invalid holiday values:",
    invalid_holiday_values,
)

Holiday values: [0, 1]
Invalid holiday values: set()


### Holiday-Flag Interpretation

The `holiday_flag` column contains only the valid binary values 0 and 1.

No correction is required.

## Numerical Validation

Numerical variables are checked for values that would be logically impossible or highly suspicious.

Potential outliers will not be removed automatically because unusual values may represent genuine business events.

In [48]:
# Checking impossible or suspicious numerical values

validation_checks = {
    "non_positive_weekly_sales": (
        df["weekly_sales"] <= 0
    ).sum(),

    "non_positive_fuel_price": (
        df["fuel_price"] <= 0
    ).sum(),

    "non_positive_cpi": (
        df["cpi"] <= 0
    ).sum(),

    "negative_unemployment": (
        df["unemployment"] < 0
    ).sum(),
}

pd.Series(
    validation_checks,
    name="invalid_count",
)

non_positive_weekly_sales    0
non_positive_fuel_price      0
non_positive_cpi             0
negative_unemployment        0
Name: invalid_count, dtype: int64

In [49]:
# Checking the temperature range

print(
    "Minimum temperature:",
    df["temperature"].min(),
)

print(
    "Maximum temperature:",
    df["temperature"].max(),
)

Minimum temperature: -2.06
Maximum temperature: 100.14


### Numerical Validation Interpretation

The numerical validation checks returned **0 invalid values** for the following:

- Non-positive weekly sales
- Non-positive fuel prices
- Non-positive CPI values
- Negative unemployment values

This confirms that these variables do not contain logically impossible values and no correction is required at this stage.

The temperature values range from **-2.06** to **100.14**. Negative temperatures are possible, so the minimum value is valid. The maximum value is also plausible because this dataset records temperature in **degrees Fahrenheit**, not Celsius.

Therefore, the observed temperature range is reasonable and should not be treated as an error.

Extreme values will still be examined during Exploratory Data Analysis using histograms and boxplots before making any outlier-removal decision.

## Chronological Sorting

The dataset contains time-based observations.

Sorting by date and store ensures that records follow a consistent chronological order and prepares the data for later time-based splitting.

In [50]:
# Sorting the data chronologically

df = (
    df
    .sort_values(
        by=["date", "store"]
    )
    .reset_index(drop=True)
)

df.head()

,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566


In [51]:
# Confirming the chronological order

print("Earliest date:", df["date"].min())
print("Latest date:", df["date"].max())

print(
    "Dates are sorted:",
    df["date"].is_monotonic_increasing,
)


Earliest date: 2010-02-05 00:00:00
Latest date: 2012-10-26 00:00:00
Dates are sorted: True


### Chronological Sorting Interpretation

The dataset has been successfully sorted in chronological order using the `date` column and then the `store` column.

The earliest record is **5 February 2010**, while the latest record is **26 October 2012**.

The result:

`Dates are sorted: True`

confirms that the `date` column is arranged from the oldest observation to the newest observation.

The first rows now show multiple stores recorded on the same earliest date, beginning with Store 1, Store 2, Store 3, and so on. This is expected because each weekly date contains observations for several Walmart stores.

Chronological sorting is important because this project contains time-based data. Later, the dataset should be split so that earlier dates are used for training and later dates are used for validation and testing. This helps prevent future information from leaking into the training data and produces a more realistic evaluation of model performance.

In [52]:
# Comparing the raw and cleaned shapes

cleaned_rows = df.shape[0]
cleaned_columns = df.shape[1]

cleaning_summary = pd.DataFrame(
    {
        "stage": [
            "Raw dataset",
            "Cleaned dataset",
        ],
        "rows": [
            initial_rows,
            cleaned_rows,
        ],
        "columns": [
            initial_columns,
            cleaned_columns,
        ],
    }
)

cleaning_summary

,stage,rows,columns
0,Raw dataset,6435,8
1,Cleaned dataset,6435,8


### Shape Comparison Interpretation

The table compares the size of the dataset before and after the data cleaning process.

| Dataset | Rows | Columns |
|---------|-----:|--------:|
| Raw Dataset | 6,435 | 8 |
| Cleaned Dataset | 6,435 | 8 |

The results show that both the raw dataset and the cleaned dataset contain **6,435 rows** and **8 columns**.

This confirms that:

- No rows were removed during the cleaning process.
- No columns were deleted.
- No duplicate records or missing values required removal.
- The cleaning phase focused on validating data quality rather than modifying the dataset.

Keeping the same dataset size is expected because previous validation steps confirmed that the data contained:
- No missing values.
- No duplicate records.
- No invalid numerical values.
- Valid date values.

Therefore, the dataset is clean, consistent, and ready for the next phase of the machine learning lifecycle: **Exploratory Data Analysis (EDA)**.

In [53]:
# Creating the processed-data folder safely

processed_data_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Processed-data directory is ready."
)

Processed-data directory is ready.


## Saving the Cleaned Dataset

The validated and cleaned dataset will now be saved in the processed-data directory.

The original raw dataset remains unchanged.

In [54]:
df.to_csv(
    processed_data_path,
    index=False,
)

print(
    "Cleaned dataset saved successfully."
)

print(
    "Saved location:",
    processed_data_path,
)

Cleaned dataset saved successfully.
Saved location: c:\Users\Udoch\walmart-sales-regression\data\procesed\walmart_cleaned.csv


In [55]:
# Confirming that the processed file exists

if not processed_data_path.exists():
    raise FileNotFoundError(
        "The cleaned dataset was not saved."
    )

print("Processed dataset verified.")

Processed dataset verified.


In [56]:
# Reloading the saved file for validation

df_cleaned_check = pd.read_csv(
    processed_data_path,
    parse_dates=["date"],
)

print(
    "Reloaded shape:",
    df_cleaned_check.shape,
)

df_cleaned_check.head()

Reloaded shape: (6435, 8)


,store,date,weekly_sales,holiday_flag,temperature,fuel_price,cpi,unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,2,2010-02-05,2136989.46,0,40.19,2.572,210.752605,8.324
2,3,2010-02-05,461622.22,0,45.71,2.572,214.424881,7.368
3,4,2010-02-05,2135143.87,0,43.76,2.598,126.442065,8.623
4,5,2010-02-05,317173.10,0,39.70,2.572,211.653972,6.566


This confirms that the saved file can be read successfully.

In [57]:
# Final validation checks
assert df_cleaned_check.shape == df.shape
assert df_cleaned_check.isna().sum().sum() == 0
assert df_cleaned_check.duplicated().sum() == 0

assert set(
    df_cleaned_check["holiday_flag"].unique()
).issubset({0, 1})

print("All final validation checks passed.")

All final validation checks passed.


## Cleaning Summary

The following cleaning and validation tasks were completed:

- A fresh copy of the raw dataset was loaded.
- Column names were standardised to lowercase snake case.
- The `date` column was converted to datetime.
- Required columns were validated.
- Missing values were checked.
- Exact duplicate records were checked.
- Duplicate store-date combinations were checked.
- Store identifiers were validated.
- Holiday flag values were validated.
- Impossible numerical values were checked.
- The dataset was sorted chronologically.
- The cleaned dataset was saved in `data/processed/`.
- The saved file was reloaded and validated successfully.

## Phase Conclusion

The Walmart sales dataset has been successfully cleaned and validated.

No rows required removal because the data contained no missing values, invalid dates, or duplicate records.

The final cleaned dataset contains 6,435 rows and 8 columns and is stored at:

`data/processed/walmart_cleaned.csv`

The next phase is **Exploratory Data Analysis**, where visualisations and statistical analysis will be used to investigate:

- Weekly-sales distribution
- Potential outliers
- Sales differences across stores
- Holiday effects
- Time-based sales patterns
- Relationships between economic variables and weekly sales
- Correlations between numerical features

In [58]:
print("Exact path:", processed_data_path)
print("File exists:", processed_data_path.exists())
print("Filename:", processed_data_path.name)
print("Parent folder:", processed_data_path.parent)
print("Files in processed folder:", list(processed_data_path.parent.iterdir()))

Exact path: c:\Users\Udoch\walmart-sales-regression\data\procesed\walmart_cleaned.csv
File exists: True
Filename: walmart_cleaned.csv
Parent folder: c:\Users\Udoch\walmart-sales-regression\data\procesed
Files in processed folder: [WindowsPath('c:/Users/Udoch/walmart-sales-regression/data/procesed/walmart_cleaned.csv')]


In [59]:
processed_data_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df.to_csv(
    processed_data_path,
    index=False,
)

print(processed_data_path.resolve())
print(processed_data_path.exists())

C:\Users\Udoch\walmart-sales-regression\data\procesed\walmart_cleaned.csv
True
